## 2026-07-30: Clean up MegaZarr's metadata schema

Load per-cohort obs as lazy frames; all transforms stay lazy until collect()

In [1]:
import hisepy as hp
import polars as pl

base = "gs://imm-zarr-poc/MegaZarr/MEGAZARR_dataset_obs"
names = ["ALTRA", "BRI", "COVID", "MM", "UP1"]
obs = {n: pl.scan_parquet(f"{base}/{n}.parquet") for n in names}

COVID parquet contains BRI samples (BR1, BR2) — keep only FH3 (COVID cohort)

In [2]:
obs["COVID"] = obs["COVID"].filter(~pl.col("cohort_cohortGuid").is_in(["BR1", "BR2"]))

- ALTRA uses PB prefix for sample IDs; kit IDs use KT — strip trailing pool suffix
- bad_kits: 9 kits present in obs but absent from HISE metadata CSV, no subject info available
- ALTRA parquet has no subject/cohort metadata — join from HISE CSV via kit ID

In [3]:
obs["ALTRA"] = obs["ALTRA"].rename({"AIFI_L3_new": "AIFI_L4"})
obs["ALTRA"] = obs["ALTRA"].with_columns(
  pl.col("pbmc_sample_id").cast(pl.String).str.replace("PB", "KT").str.split("-").list.first().alias("sample.sampleKitGuid")
)
bad_kits = {'KT00473', 'KT00069', 'KT00076', 'KT00085', 'KT04655', 'KT04110', 'KT00496', 'KT02946', 'KT00784'}
obs["ALTRA"] = obs["ALTRA"].filter(pl.col("sample.sampleKitGuid").is_in(bad_kits).not_())

meta_path = hp.reader.cache_files(['242128d8-4449-4b03-a00c-d9729a511437'])
metadata = pl.read_csv('/home/workspace/input/3974417942/UCSDCU_Y4/242128d8-4449-4b03-a00c-d9729a511437/metadata/CU_SD_729_blood_sample_meta_lab_data_clean_20260414.csv', 
                       null_values=["NA", ""], ignore_errors=True)

meta_subset = (
  metadata
  .select(["sampleKitGuid", "subjectGuid", "cohort", "id"])
  .unique(subset=["sampleKitGuid"])
  .rename({
      "sampleKitGuid": "sample.sampleKitGuid",
      "subjectGuid":   "subject.subjectGuid",
      "cohort":        "cohort.cohortGuid",
      "id":            "pipeline.fileGuid"
  })
)

obs["ALTRA"] = obs["ALTRA"].join(meta_subset.lazy(), on="sample.sampleKitGuid", how="left")

2026-08-03 16:08:06,369 INFO [hisepy.logging:185] logging 3111 137759837464384 Calling cache_files
2026-08-03 16:08:12,120 INFO [hisepy.logging:228] logging 3111 137759837464384 Finished cache_files successfully (time_elapsed=3.259s)


- MM parquet also contains BRI samples (BR1, BR2) — keep only FH1 (MM cohort)
- MM parquet missing pipeline.fileGuid — sourced from scrna_metadata.csv via kit ID

In [4]:
scrna_meta = pl.read_csv("files/scrna_metadata.csv", ignore_errors=True)
file_map = (
  scrna_meta
  .select(["sample.sampleKitGuid", "file.id"])
  .unique(subset=["sample.sampleKitGuid"])
  .with_columns(pl.col("sample.sampleKitGuid").cast(pl.Categorical))
  .lazy()
)

obs["MM"] = (
  obs["MM"]
  .filter(pl.col("cohort.cohortGuid") == "FH1")
  .join(file_map, on="sample.sampleKitGuid", how="left")
  .rename({
      "file.id":        "pipeline.fileGuid",
      "subject.age":    "sample.subjectAgeAtDraw",
      "aifi_label_l1":  "AIFI_L1",
      "aifi_label_l2":  "AIFI_L2",
      "aifi_label_l3":  "AIFI_L3",
  })
)

# verify cell type cols landed correctly
print([c for c in obs["MM"].collect_schema().names() if "aifi" in c.lower() or c.startswith("AIFI")])

/home/workspace/environment/minimalv4/lib/python3.11/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


['aifi_l1', 'aifi_l2', 'aifi_l3', 'AIFI_L1', 'aifi_celltype_l1', 'AIFI_L2', 'aifi_celltype_l2', 'aifi_plot_l2', 'AIFI_L3', 'aifi_celltype_l3', 'aifi_plot_l3']


- ALTRA has a 4th label tier (AIFI_L4); all other cohorts stop at L3

In [5]:
ct_keep = {
    "ALTRA": {"AIFI_L3", "AIFI_L4"},
    "BRI":   {"AIFI_L3"},
    "COVID": {"AIFI_L3"},
    "MM":    {"AIFI_L3"},
    "UP1":   {"AIFI_L3"},
}

ct_all = {
    "AIFI_L1", "AIFI_L1_score", "AIFI_L2", "AIFI_L2_score",
    "AIFI_L3", "AIFI_L3_score", "AIFI_L4",
    "predicted_AIFI_L1", "predicted_AIFI_L2", "predicted_AIFI_L3",
    "aifi_l1", "aifi_l2", "aifi_l3",
    "aifi_celltype_l1", "aifi_celltype_l2", "aifi_celltype_l3",
    "aifi_plot_l2", "aifi_plot_l3",
}

for name in obs:
    schema_cols = set(obs[name].collect_schema().names())
    obs[name] = obs[name].drop(list((ct_all - ct_keep[name]) & schema_cols))

- Regenerate L1/L2 from L3 via tidy-labels to ensure consistent labels across cohorts
- Drop any existing L1/L2 first — some cohorts had these but they may be inconsistent

In [6]:
labels = (
    pl.read_csv("files/tidy-labels.csv")
    .select([
        pl.col("l3_label").str.strip_chars().cast(pl.Categorical).alias("AIFI_L3"),
        pl.col("l2_label").str.strip_chars().alias("AIFI_L2"),
        pl.col("l1_label").str.strip_chars().alias("AIFI_L1"),
    ])
    .unique()
    .lazy()
)

for name in obs:
    have = obs[name].collect_schema().names()
    obs[name] = obs[name].drop([c for c in ["AIFI_L1", "AIFI_L2"] if c in have])
    obs[name] = obs[name].join(labels, on="AIFI_L3", how="left")

- tidy-labels has "CD8aa T cell" but data uses "CD8aa" — patch manually rather than rename data values

In [7]:
for name in obs:
    obs[name] = obs[name].with_columns([
        pl.when(pl.col("AIFI_L3") == "CD8aa").then(pl.lit("T cell")).otherwise(pl.col("AIFI_L1")).alias("AIFI_L1"),
        pl.when(pl.col("AIFI_L3") == "CD8aa").then(pl.lit("CD8aa")).otherwise(pl.col("AIFI_L2")).alias("AIFI_L2"),
    ])

- COVID had both pipeline_fileGuid and file_id pointing to the same data — drop one before renaming
- Normalize underscore-separated column names to dot notation across all cohorts

In [8]:
obs["COVID"] = obs["COVID"].drop("pipeline_fileGuid")

normalize = {
    "cohort_cohortGuid":       "cohort.cohortGuid",
    "pipeline.fileGuid":       "file.id",
    "sample_drawDate":         "sample.drawDate",
    "sample_sampleKitGuid":    "sample.sampleKitGuid",
    "sample_subjectAgeAtDraw": "sample.subjectAgeAtDraw",
    "sample_visitName":        "sample.visitName",
    "specimen_specimenGuid":   "specimen.specimenGuid",
    "subject_ageAtFirstDraw":  "subject.ageAtFirstDraw",
    "subject_ageGroup":        "subject.ageGroup",
    "subject_biologicalSex":   "subject.biologicalSex",
    "subject_birthYear":       "subject.birthYear",
    "subject_bmi":             "subject.bmi",
    "subject_cmv":             "subject.cmv",
    "subject_ethnicity":       "subject.ethnicity",
    "subject_race":            "subject.race",
    "subject_subjectGuid":     "subject.subjectGuid",
    "specimens.specimenGuid":  "specimen.specimenGuid",
    "specimens.specimenType":  "specimen.specimenType",
    "specimens_specimenType":  "specimen.specimenType",
    "file_id":                 "file.id"
}

for name in obs:
    have = obs[name].collect_schema().names()
    renames = {k: v for k, v in normalize.items() if k in have}
    if renames:
        obs[name] = obs[name].rename(renames)

In [9]:
to_drop = [
  # cohort artifacts
  "pbmc_sample_id", "index", "source_file", "dataset", "cohort_ID",
  # duplicate mito
  "pct_counts_mt", "total_counts_mt", "log1p_total_counts_mt",
  # derived QC
  "log1p_n_genes_by_counts", "log1p_total_counts", "log1p_total_counts_mito",
  "log1p_total_counts_hb", "log1p_total_counts_ribo",
  "pct_counts_in_top_20_genes", "pct_counts_in_top_50_genes",
  "pct_counts_in_top_100_genes", "pct_counts_in_top_200_genes",
  "pct_counts_in_top_500_genes", "n_genes_by_counts",
  # MM clustering/embedding
  "leiden", "leiden_2", "leiden_harmony_2", "umap_1", "umap_2",
  "seurat_pbmc_type", "seurat_pbmc_type_score",
  # MM visit label duplicates
  "label.visitDetails", "label.visitName",
  # MM manual annotations
  "manual.batch_id", "manual.category", "manual.extracted_name",
  "manual.flu_response", "manual.response", "manual.response_type",
  "manual.time_stamp", "manual.treatment_dara",

  "pct_counts_hb", "total_counts_hb",        # COVID only
  "pct_counts_ribo", "total_counts_ribo",    # COVID only
  "outlier", "mt_outlier",                   # COVID only
  "subject.ageGroup",                        # derived
  "subject.ageAtFirstDraw"                   # less precise than subjectAgeAtDraw
]

for name in obs:
  have = obs[name].collect_schema().names()
  obs[name] = obs[name].drop([c for c in to_drop if c in have])

for name in obs:
  df = obs[name].collect()
  all_null = [col for col in df.columns if df[col].is_null().all()]
  if all_null:
      print(f"{name}: dropping {all_null}")
      obs[name] = obs[name].drop(all_null)

# Unify draw date: BRI sample.drawYear → sample.drawDate
obs["BRI"] = obs["BRI"].rename({"sample.drawYear": "sample.drawDate"})

COVID: dropping ['subject.cmv', 'subject.bmi', 'sample.drawDate', 'sample.subjectAgeAtDraw', 'specimen.specimenGuid']


In [10]:
# BRI: cast all String cols to Categorical
bri_to_cat = [
  "batch_id", "chip_id", "pool_id", "well_id",
  "cohort.cohortGuid", "sample.sampleKitGuid", "sample.visitName",
  "specimen.specimenGuid", "subject.biologicalSex", "subject.cmv",
  "subject.ethnicity", "subject.race", "subject.subjectGuid", "file.id",
]
obs["BRI"] = obs["BRI"].with_columns(
  [pl.col(c).cast(pl.Categorical) for c in bri_to_cat]
)

# ALTRA: cast remaining String cols to Categorical
obs["ALTRA"] = obs["ALTRA"].with_columns([
  pl.col("cohort.cohortGuid").cast(pl.Categorical),
  pl.col("sample.sampleKitGuid").cast(pl.Categorical),
  pl.col("subject.subjectGuid").cast(pl.Categorical),
])

# COVID: n_mito_umis is Float64, others are UInt16 — cast down
obs["COVID"] = obs["COVID"].with_columns(
  pl.col("n_mito_umis").cast(pl.UInt16)
)

# BRI: sample.drawDate is Int64 (year only) — cast to String so concat doesn't fail
obs["BRI"] = obs["BRI"].with_columns(
  pl.col("sample.drawDate").cast(pl.String).cast(pl.Categorical)
)

In [11]:
# Rename file.id back to pipeline.fileGuid
for name in obs:
  if "file.id" in obs[name].collect_schema().names():
      obs[name] = obs[name].rename({"file.id": "pipeline.fileGuid"})

# Canonical columns to keep
canonical = [
  # process identifiers
  "barcodes", "original_barcodes", "cell_name",
  "batch_id", "pool_id", "chip_id", "well_id",
  # QC
  "n_reads", "n_umis", "n_genes",
  # cell type labels
  "AIFI_L1", "AIFI_L2", "AIFI_L3", "AIFI_L4",
  # sample identifiers
  "cohort.cohortGuid", "subject.subjectGuid", "sample.sampleKitGuid",
  "specimen.specimenGuid", "pipeline.fileGuid"
]

for name in obs:
  have = obs[name].collect_schema().names()
  obs[name] = obs[name].select([c for c in canonical if c in have])

- re-merge data quality check — only prints columns with nulls or zeros

In [12]:
numeric_types = {pl.Int8, pl.Int16, pl.Int32, pl.Int64, pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, pl.Float32, pl.Float64}

for name, lf in obs.items():
  df = lf.collect()
  n = df.shape[0]
  print(f"\n{name} ({n:,} cells):")
  for col in df.columns:
      null_cnt = df[col].null_count()
      zero_cnt = int((df[col] == 0).sum()) if df[col].dtype in numeric_types else 0
      if null_cnt > 0 or zero_cnt > 0:
          print(f"  {col}: {null_cnt:,} null  {zero_cnt:,} zero")


ALTRA (7,846,472 cells):

BRI (13,789,548 cells):

COVID (2,389,878 cells):

MM (2,374,117 cells):

UP1 (3,541,330 cells):


- diagonal_relaxed fills missing columns with null rather than erroring on schema mismatch

In [13]:
merged = pl.concat(list(obs.values()), how="diagonal_relaxed")
print(merged.collect_schema().names())

df = merged.collect()
print(df.shape)

df.write_parquet("gs://imm-zarr-poc/MegaZarr/MEGAZARR_dataset_obs/megazarr_cleaned.parquet")

['barcodes', 'original_barcodes', 'cell_name', 'batch_id', 'pool_id', 'chip_id', 'well_id', 'n_reads', 'n_umis', 'n_genes', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'AIFI_L4', 'cohort.cohortGuid', 'subject.subjectGuid', 'sample.sampleKitGuid', 'pipeline.fileGuid', 'specimen.specimenGuid']
(29941345, 19)
